# SRISK Calculation - Interactive Debug Notebook

This notebook contains the main SRISK calculation workflow from `scripts/04_calculate_srisk.py`.
Run cells step-by-step to debug and inspect intermediate results.

**Pipeline:**
1. Load processed data (aligned debt + equity)
2. Load LRMES estimates
3. Calculate SRISK
4. System aggregates
5. Summary statistics
6. Country aggregates
7. Decomposition analysis

## Setup

In [2]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data import load_dataframe, save_dataframe
from src.models.srisk import (
    calculate_srisk_multi,
    calculate_system_srisk,
    calculate_srisk_contribution,
    srisk_summary_statistics,
    calculate_srisk_by_country,
    decompose_srisk
)
from src.utils.config import CONFIG, PROCESSED_DATA_DIR, RESULTS_DATA_DIR
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ Imports successful")
print(f"\nConfiguration:")
print(f"  Capital ratio: {CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%")
print(f"  Crisis threshold: {CONFIG.CRISIS_THRESHOLD*100:.1f}%")
print(f"  Horizon: {CONFIG.CRISIS_HORIZON_WEEKS} weeks")

✓ Imports successful

Configuration:
  Capital ratio: 8.0%
  Crisis threshold: -15.0%
  Horizon: 22 weeks


## Step 1: Load Data

In [3]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

# Load aligned data
aligned_data = load_dataframe("aligned_daily_data", PROCESSED_DATA_DIR)
print(f"✓ Aligned data: {aligned_data.shape}")
print(f"  Date range: {aligned_data.index.min()} to {aligned_data.index.max()}")

# Load LRMES estimates
lrmes_df = load_dataframe("lrmes_estimates", RESULTS_DATA_DIR)
print(f"\n✓ LRMES estimates: {len(lrmes_df)} banks")

# Load bank universe
bank_universe = load_dataframe("banks_universe_clean", PROCESSED_DATA_DIR)
print(f"✓ Bank universe: {len(bank_universe)} banks")

LOADING DATA
✓ Loaded aligned_daily_data.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/processed
  Shape: (3772, 287)
✓ Aligned data: (3772, 287)
  Date range: 2009-12-29 00:00:00 to 2026-03-09 00:00:00
✓ Loaded lrmes_estimates.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/results
  Shape: (40, 12)

✓ LRMES estimates: 40 banks
✓ Loaded banks_universe_clean.parquet from /Users/milanszabo/PYTHON_PROJECTS/SRISK_GULF/data/processed
  Shape: (40, 3)
✓ Bank universe: 40 banks


In [4]:
# Inspect LRMES estimates
print("LRMES Estimates (top 10 by LRMES):")
display(lrmes_df.sort_values('lrmes', ascending=False).head(10))

LRMES Estimates (top 10 by LRMES):


,bank_ric,lrmes,n_crisis_scenarios,crisis_probability,avg_bank_loss_in_crisis,avg_market_loss_in_crisis,garch_persistence,dcc_a,dcc_b,dcc_persistence,avg_correlation,convergence
36,1050.SE,0.0936,346,0.0046,0.1016,0.1993,0.9763,0.1796,0.0000,0.1796,0.4346,True
33,1150.SE,0.0936,346,0.0046,0.1029,0.1993,0.9889,0.1531,0.7850,0.9381,0.4704,True
32,1120.SE,0.0902,346,0.0046,0.0971,0.1993,0.9236,0.1643,0.7267,0.8910,0.5538,True
31,1020.SE,0.0820,346,0.0046,0.0879,0.1993,0.8997,0.1574,0.8416,0.9990,0.4604,True
34,1080.SE,0.0808,346,0.0046,0.0864,0.1993,0.9325,0.1986,0.7073,0.9058,0.4131,True
37,1010.SE,0.0771,346,0.0046,0.0825,0.1993,0.9946,0.0889,0.3569,0.4458,0.4369,True
35,1140.SE,0.0764,346,0.0046,0.0826,0.1993,0.9199,0.1624,0.7622,0.9246,0.4269,True
38,1060.SE,0.0739,346,0.0046,0.0787,0.1993,0.9635,0.1795,0.7630,0.9425,0.4069,True
7,ADCB.AD,0.0614,346,0.0046,0.0686,0.1993,0.9554,0.2015,0.7232,0.9247,0.2662,True
15,QISB.QA,0.0608,346,0.0046,0.0677,0.1993,0.9777,0.1960,0.0000,0.1960,0.2668,True


In [7]:
# Convert LRMES to time series (broadcast to all dates)
print("Converting LRMES to time series...")

lrmes_values = dict(zip(lrmes_df["bank_ric"], lrmes_df["lrmes"]))

# Create time series DataFrame
lrmes_ts = pd.DataFrame(
    {inst: [lrmes_values[inst]] * len(aligned_data) for inst in lrmes_values.keys()},
    index=aligned_data.index
)

print(f"✓ LRMES time series: {lrmes_ts.shape}")
print(f"  Columns: {list(lrmes_ts.columns[:5])} ...")

display(lrmes_ts)

Converting LRMES to time series...
✓ LRMES time series: (3772, 40)
  Columns: ['AJBNK.DU', 'DISB.DU', 'ENBD.DU', 'ADIB.AD', 'FAB.AD'] ...


,AJBNK.DU,DISB.DU,ENBD.DU,ADIB.AD,FAB.AD,BOS.AD,SIB.AD,ADCB.AD,BBKB.BH,SALAM.BH,NATB.BH,DOBK.QA,MARK.QA,COMB.QA,QIIB.QA,QISB.QA,QNBK.QA,ABOB.OM,BKDB.OM,BKMB.OM,BKNZ.OM,NBOB.OM,BKSB.OM,KFH.KW,ABKK.KW,BOUK.KW,NBKK.KW,BURG.KW,KIBK.KW,KPRO.KW,GBKK.KW,1020.SE,1120.SE,1150.SE,1080.SE,1140.SE,1050.SE,1010.SE,1060.SE,1030.SE
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2009-12-29,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
2009-12-30,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
2009-12-31,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
2010-01-03,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
2010-01-04,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-02,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
2026-03-04,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589
2026-03-05,0.0273,0.0586,0.0436,0.0270,0.0411,0.0137,0.0190,0.0614,0.0073,0.0368,0.0000,0.0321,0.0447,0.0425,0.0408,0.0608,0.0359,0.0108,0.0202,0.0142,0.0173,0.0106,0.0230,0.0123,0.0184,0.0152,0.0357,0.0218,0.0299,0.0207,0.0190,0.0820,0.0902,0.0936,0.0808,0.0764,0.0936,0.0771,0.0739,0.0589


## Step 2: Calculate SRISK

In [6]:
print("=" * 60)
print("CALCULATING SRISK")
print("=" * 60)
print(f"Capital ratio: {CONFIG.CAPITAL_RATIO_BASEL * 100:.1f}%")
print()

# Calculate SRISK
srisk = calculate_srisk_multi(
    aligned_data=aligned_data,
    lrmes_df=lrmes_ts,
    bank_universe=bank_universe,
    k=CONFIG.CAPITAL_RATIO_BASEL
)

print(f"✓ SRISK calculated for {len(srisk.columns)} banks")
print(f"  Shape: {srisk.shape}")

CALCULATING SRISK
Capital ratio: 8.0%

⚠ Skipping AJBNK.DU: Missing debt or equity data
⚠ Skipping DISB.DU: Missing debt or equity data
⚠ Skipping ENBD.DU: Missing debt or equity data
⚠ Skipping ADIB.AD: Missing debt or equity data
⚠ Skipping FAB.AD: Missing debt or equity data
⚠ Skipping BOS.AD: Missing debt or equity data
⚠ Skipping SIB.AD: Missing debt or equity data
⚠ Skipping ADCB.AD: Missing debt or equity data
⚠ Skipping BBKB.BH: Missing debt or equity data
⚠ Skipping SALAM.BH: Missing debt or equity data
⚠ Skipping NATB.BH: Missing debt or equity data
⚠ Skipping DOBK.QA: Missing debt or equity data
⚠ Skipping MARK.QA: Missing debt or equity data
⚠ Skipping COMB.QA: Missing debt or equity data
⚠ Skipping QIIB.QA: Missing debt or equity data
⚠ Skipping QISB.QA: Missing debt or equity data
⚠ Skipping QNBK.QA: Missing debt or equity data
⚠ Skipping ABOB.OM: Missing debt or equity data
⚠ Skipping BKDB.OM: Missing debt or equity data
⚠ Skipping BKMB.OM: Missing debt or equity data
⚠ 

In [ ]:
# Inspect SRISK - latest values
print("Latest SRISK (top 10 banks):")
latest_srisk = srisk.iloc[-1].sort_values(ascending=False)
print(latest_srisk.head(10).apply(lambda x: f"${x/1e9:.2f}B"))

In [ ]:
# Save SRISK time series
save_dataframe(srisk, "srisk_timeseries", RESULTS_DATA_DIR, {
    "description": f"SRISK time series (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL,
    "crisis_threshold": CONFIG.CRISIS_THRESHOLD,
    "horizon_weeks": CONFIG.CRISIS_HORIZON_WEEKS
})
print("✓ Saved: srisk_timeseries.parquet")

## Step 3: System Aggregates

In [ ]:
print("=" * 60)
print("SYSTEM-WIDE AGGREGATION")
print("=" * 60)

# System SRISK (sum of positive SRISK)
system_srisk = calculate_system_srisk(srisk)

print(f"\nSystem SRISK (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%):")
print(f"  Latest: ${system_srisk.iloc[-1] / 1e9:.2f}B")
print(f"  Mean: ${system_srisk.mean() / 1e9:.2f}B")
print(f"  Max: ${system_srisk.max() / 1e9:.2f}B")
print(f"  Min: ${system_srisk.min() / 1e9:.2f}B")

In [ ]:
# Plot system SRISK over time
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(system_srisk.index, system_srisk / 1e9)
plt.title('System-wide SRISK Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('SRISK ($ Billions)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate contributions (percentage of system SRISK)
contributions = calculate_srisk_contribution(srisk)

print(f"\nTop 5 contributors (latest):")
latest_contrib = contributions.iloc[-1].sort_values(ascending=False).head()
for bank, pct in latest_contrib.items():
    print(f"  - {bank}: {pct:.2f}%")

In [ ]:
# Save system aggregates
save_dataframe(system_srisk, "system_srisk", RESULTS_DATA_DIR, {
    "description": "System-wide SRISK (sum of positive SRISK)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

save_dataframe(contributions, "srisk_contributions", RESULTS_DATA_DIR, {
    "description": "Bank contributions to system SRISK (%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: system_srisk.parquet")
print("✓ Saved: srisk_contributions.parquet")

## Step 4: Summary Statistics

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

# Summary statistics
summary = srisk_summary_statistics(srisk, bank_universe)

print(f"\nTop 10 banks by mean SRISK:")
display(summary.head(10)[["bank_ric", "country", "mean_srisk", "latest_srisk", "max_srisk"]])

In [ ]:
# Save summary
save_dataframe(summary, "srisk_summary", RESULTS_DATA_DIR, {
    "description": f"SRISK summary statistics (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: srisk_summary.parquet")

## Step 5: Country Aggregates

In [ ]:
print("=" * 60)
print("COUNTRY AGGREGATION")
print("=" * 60)

country_srisk = calculate_srisk_by_country(srisk, bank_universe)

print(f"\nSRISK by country (latest):")
latest_country = country_srisk.iloc[-1].sort_values(ascending=False)
for country, srisk_val in latest_country.items():
    print(f"  - {country}: ${srisk_val / 1e9:.2f}B")

In [ ]:
# Plot country SRISK
plt.figure(figsize=(10, 6))
latest_country.plot(kind='barh')
plt.title('SRISK by Country (Latest)', fontsize=14, fontweight='bold')
plt.xlabel('SRISK ($ Billions)')
plt.ylabel('Country')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Save country aggregates
save_dataframe(country_srisk, "srisk_by_country", RESULTS_DATA_DIR, {
    "description": f"SRISK aggregated by country (k={CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: srisk_by_country.parquet")

## Step 6: Decomposition Analysis

In [ ]:
print("=" * 60)
print("DECOMPOSITION ANALYSIS")
print("=" * 60)
print("Analyzing top 5 banks by latest SRISK...")

# Identify top banks
top_banks = srisk.iloc[-1].sort_values(ascending=False).head(5).index.tolist()
print(f"\nTop 5 banks: {top_banks}")

In [ ]:
# Decompose SRISK for each top bank
decomp_results = []

for bank_ric in top_banks:
    debt_col = f"fund_{bank_ric}_TotLiab"
    equity_col = f"mktcap_{bank_ric}"
    
    if debt_col in aligned_data.columns and equity_col in aligned_data.columns:
        debt = aligned_data[debt_col]
        equity = aligned_data[equity_col]
        lrmes = lrmes_ts[bank_ric]
        
        decomp = decompose_srisk(debt, equity, lrmes, k=CONFIG.CAPITAL_RATIO_BASEL)
        
        # Summarize
        decomp_results.append({
            "bank_ric": bank_ric,
            "avg_size_effect": decomp["size_effect"].mean(),
            "avg_leverage_effect": decomp["leverage_effect"].mean(),
            "avg_risk_effect": decomp["risk_effect"].mean(),
            "total_change": decomp["srisk"].iloc[-1] - decomp["srisk"].iloc[0]
        })

decomp_df = pd.DataFrame(decomp_results)
print(f"\nDecomposition (average daily effects):")
display(decomp_df)

In [ ]:
# Save decomposition
save_dataframe(decomp_df, "srisk_decomposition", RESULTS_DATA_DIR, {
    "description": "SRISK decomposition (size, leverage, risk effects)",
    "capital_ratio": CONFIG.CAPITAL_RATIO_BASEL
})

print("✓ Saved: srisk_decomposition.parquet")

## Final Summary

In [ ]:
print("=" * 60)
print("✓ SRISK CALCULATION COMPLETED")
print("=" * 60)
print(f"\nCapital ratio used: {CONFIG.CAPITAL_RATIO_BASEL*100:.1f}%")
print(f"Crisis threshold: {CONFIG.CRISIS_THRESHOLD*100:.1f}% market decline")
print(f"Horizon: {CONFIG.CRISIS_HORIZON_WEEKS} weeks")
print(f"\nResults saved to: {RESULTS_DATA_DIR}")
print("\nKey outputs:")
print("  - srisk_timeseries.parquet - SRISK time series")
print("  - system_srisk.parquet - System-wide SRISK")
print("  - srisk_contributions.parquet - Bank contributions (%)")
print("  - srisk_summary.parquet - Summary statistics")
print("  - srisk_by_country.parquet - Country aggregates")
print("  - srisk_decomposition.parquet - Decomposition analysis")